# Cross-dataset evaluation of the URL-Phish-trained pipelines


In [1]:
!pip install -q category_encoders tldextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 2.9 MB/s eta 0:00:00


## 1. Config 

In [2]:
import os, glob, json, math, ipaddress, warnings, time
from collections import Counter
from urllib.parse import urlparse
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---- training data (URL-Phish) ----
TRAIN_CSV = None            # None -> auto-detect Dataset.csv under /kaggle/input
# ---- external datasets (Kaggle inputs; each folder is searched recursively for csv / parquet / json files) ----
EXT_SETS = {
    "Hannousse2020": {"dir": "/kaggle/input/datasets/ummehabibamoushi/hannousse-phishing",
                      "max_rows": 20000, "prefer": None},      # whole dataset (11,430 URLs)
    "PhreshPhish":   {"dir": "/kaggle/input/datasets/ummehabibamoushi/pheshphish-dataset",
                      "max_rows": 30000, "prefer": "test"},    # uses files with 'test' in the name if present
}
FLIP = {}          # set e.g. {"PhreshPhish": True} ONLY if the notebook warns that a dataset's labels look inverted
# ---- run options ----
MODELS = ["SVM", "LR", "DT", "RF", "Stacking", "NB"]          # SVM is the paper's main model; trim the list to go faster
TRAIN_FEATURES = "auto"     # "auto": use the CSV's precomputed columns if the extractor reproduces them (>=99%), else recompute them
                            # "dataset" / "recomputed" force one of the two (recomputed = identical extractor code for train and external)
SEED = 42
OUT = "/kaggle/working/cross_dataset_results" if os.path.exists("/kaggle/working") else "cross_dataset_results"
os.makedirs(OUT + "/models", exist_ok=True)
CAT_COLS = ["dom", "tld"]
BASE_RATES = [0.144, 0.05, 0.01]

## 2. Feature extractor and **calibration against your own CSV**


In [3]:
import tldextract
from sklearn.base import BaseEstimator, TransformerMixin
_EXT = tldextract.TLDExtract(suffix_list_urls=())      # offline snapshot, no network needed

def _split(url):
    u = str(url).strip()
    try:
        p = urlparse(u if "://" in u else "//" + u); host = (p.hostname or "").lower()
    except ValueError:
        p = urlparse(""); host = ""
    return u, p, host, (_EXT(host) if host else _EXT(""))

def _entropy(s):
    if not s: return 0.0
    c, n = Counter(s), len(s)
    return float(-sum(v / n * math.log2(v / n) for v in c.values()))

def _is_ip(h):
    try: ipaddress.ip_address(h); return 1
    except ValueError: return 0

def url_parts(url):
    """registrable domain and public suffix, same logic used for `dom` / `tld` of external rows"""
    u, p, host, e = _split(url)
    return (e.registered_domain or host), e.suffix

def _cands(url):
    u, p, host, e = _split(url)
    n = max(len(u), 1); dom = e.registered_domain or host
    sub_labels = [s for s in e.subdomain.split(".") if s]
    path, query = p.path or "", p.query or ""
    noscheme = u.split("://", 1)[1] if "://" in u else u
    letters, digits = sum(c.isalpha() for c in u), sum(c.isdigit() for c in u)
    sp = {"non_alnum": sum(not c.isalnum() for c in u),
          "non_alnum_excl_dot": sum((not c.isalnum()) and c != "." for c in u),
          "non_alnum_excl_dot_slash_colon": sum((not c.isalnum()) and c not in "./:" for c in u)}
    o = {"url_len|len": len(u),
         "dom_len|registered": len(dom), "dom_len|hostname": len(host),
         "is_ip|host_is_ip": _is_ip(host),
         "tld_len|suffix": len(e.suffix),
         "subdom_cnt|labels": len(sub_labels), "subdom_cnt|labels_excl_www": len([s for s in sub_labels if s != "www"]),
         "dot_cnt|url": u.count("."), "dot_cnt|host": host.count("."),
         "dash_cnt|url": u.count("-"), "dash_cnt|host": host.count("-"),
         "under_cnt|url": u.count("_"),
         "slash_cnt|url": u.count("/"), "slash_cnt|path": path.count("/"),
         "qm_cnt|url": u.count("?"), "eq_cnt|url": u.count("="), "amp_cnt|url": u.count("&"),
         "digit_cnt|url": digits, "letter_cnt|url": letters,
         "digit_ratio|url": digits / n, "letter_ratio|url": letters / n,
         "is_https|startswith": int(u.lower().startswith("https")),
         "entropy|url": _entropy(u), "entropy|no_scheme": _entropy(noscheme), "entropy|host": _entropy(host),
         "path_len|path": len(path), "path_len|path_query": len(path) + len(query),
         "query_len|query": len(query)}
    for k, v in sp.items():
        o["special_cnt|" + k] = v; o["spec_ratio|" + k] = v / n
    return o

def cand_frame(urls): return pd.DataFrame([_cands(u) for u in urls])

In [4]:
# ---- load + clean training data exactly like the paper notebook ----
if TRAIN_CSV is None:
    hits = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    TRAIN_CSV = next((f for f in hits if os.path.basename(f).lower() == "dataset.csv"), hits[0])
print("train csv:", TRAIN_CSV)
df = pd.read_csv(TRAIN_CSV)
TLD_MODE = df["tld"].mode()[0]
df["tld"] = df["tld"].fillna(TLD_MODE)
df = df.drop_duplicates().reset_index(drop=True)
y = df["label"].astype(int)
NUM = [c for c in df.columns if c not in ("url", "dom", "tld", "label")]
print(len(df), "rows |", int(y.sum()), "phishing |", len(NUM), "numeric features:", NUM)

# ---- calibration: which candidate definition reproduces each precomputed column? ----
samp = df.sample(min(6000, len(df)), random_state=SEED)
cf = cand_frame(samp["url"].tolist()); cf.index = samp.index
TOL = 0.0051                      # ratios may be rounded to 2 decimals in the CSV
CHOSEN, rows = {}, []
for feat in NUM:
    cols = [c for c in cf.columns if c.split("|")[0] == feat]
    if not cols: rows.append({"feature": feat, "best": "NO CANDIDATE", "match": 0.0, "runner_up": ""}); continue
    rate = {c: float(np.isclose(cf[c].astype(float), samp[feat].astype(float), rtol=0, atol=TOL).mean()) for c in cols}
    order = sorted(rate, key=rate.get, reverse=True)
    CHOSEN[feat] = order[0]
    rows.append({"feature": feat, "best": order[0], "match": round(rate[order[0]], 4),
                 "runner_up": (f"{order[1]} ({rate[order[1]]:.3f})" if len(order) > 1 else "")})
calib = pd.DataFrame(rows); calib.to_csv(OUT + "/feature_calibration.csv", index=False)
display(calib)

# dom / tld definitions
d_reg = samp["url"].map(lambda u: url_parts(u)[0]); t_suf = samp["url"].map(lambda u: url_parts(u)[1])
print("dom == registrable domain of url:", round(float((d_reg == samp["dom"]).mean()), 4))
TLD_MATCH = float((t_suf == samp["tld"]).mean()); print("tld == public suffix of url     :", round(TLD_MATCH, 4))

missing = [f for f in NUM if f not in CHOSEN]
bad = calib[(calib.match < 0.99)]
if missing: raise RuntimeError(f"No extractor candidate for: {missing}. Add one in _cands() before continuing.")
if len(bad):
    print("\nWARNING: these features are NOT reproduced (<99% match):", bad.feature.tolist())
    print("-> Either fix _cands() for them, or set TRAIN_FEATURES='recomputed' so train and external use the SAME code.")
else:
    print("\nAll features reproduced (>=99%). Extractor is consistent with your CSV.")

def extract_features(urls):
    c = cand_frame(urls)
    return pd.DataFrame({f: c[CHOSEN[f]].to_numpy() for f in NUM})

TRAIN_FEATURES_EFF = TRAIN_FEATURES if TRAIN_FEATURES != "auto" else ("dataset" if (len(bad) == 0 and TLD_MATCH >= 0.99) else "recomputed")
print("\n>>> training features mode:", TRAIN_FEATURES_EFF, "(precomputed CSV columns)" if TRAIN_FEATURES_EFF == "dataset" else "(recomputed from url; paper-number check does not apply)")

train csv: /kaggle/input/datasets/ummehabibamoushi/hannousse-phishing/Dataset.csv
115231 rows | 16590 phishing | 22 numeric features: ['url_len', 'dom_len', 'is_ip', 'tld_len', 'subdom_cnt', 'letter_cnt', 'digit_cnt', 'special_cnt', 'eq_cnt', 'qm_cnt', 'amp_cnt', 'dot_cnt', 'dash_cnt', 'under_cnt', 'letter_ratio', 'digit_ratio', 'spec_ratio', 'is_https', 'slash_cnt', 'entropy', 'path_len', 'query_len']


,feature,best,match,runner_up
0,url_len,url_len|len,1.0,
1,dom_len,dom_len|registered,1.0,dom_len|hostname (0.202)
2,is_ip,is_ip|host_is_ip,1.0,
3,tld_len,tld_len|suffix,1.0,
4,subdom_cnt,subdom_cnt|labels,1.0,subdom_cnt|labels_excl_www (0.353)
5,letter_cnt,letter_cnt|url,1.0,
6,digit_cnt,digit_cnt|url,1.0,
7,special_cnt,special_cnt|non_alnum,1.0,special_cnt|non_alnum_excl_dot (0.000)
8,eq_cnt,eq_cnt|url,1.0,
9,qm_cnt,qm_cnt|url,1.0,


dom == registrable domain of url: 1.0
tld == public suffix of url     : 1.0

All features reproduced (>=99%). Extractor is consistent with your CSV.

>>> training features mode: dataset (precomputed CSV columns)


## 3. Trainand save the fixed pipelines

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score, balanced_accuracy_score,
                             matthews_corrcoef, roc_auc_score, average_precision_score)
import category_encoders as ce

class NestedTargetEncoder(BaseEstimator, TransformerMixin):          # identical to experiment.ipynb (grouped=True)
    def __init__(self, cat_cols, group_col="grp", n_splits=5, smoothing=10, grouped=True, seed=SEED):
        self.cat_cols, self.group_col = cat_cols, group_col
        self.n_splits, self.smoothing, self.grouped, self.seed = n_splits, smoothing, grouped, seed
    def _enc(self): return ce.TargetEncoder(cols=self.cat_cols, smoothing=self.smoothing)
    def fit(self, X, y): self.enc_ = self._enc().fit(X[self.cat_cols], y); return self
    def fit_transform(self, X, y):
        X = X.reset_index(drop=True); y = pd.Series(np.asarray(y)).reset_index(drop=True)
        out = np.zeros((len(X), len(self.cat_cols)))
        for a, b in GroupKFold(self.n_splits).split(X, y, groups=X[self.group_col]):
            out[b] = self._enc().fit(X.loc[a, self.cat_cols], y.loc[a]).transform(X.loc[b, self.cat_cols]).to_numpy()
        self.fit(X, y); return out
    def transform(self, X): return self.enc_.transform(X[self.cat_cols]).to_numpy()

def make_model(name, seed=SEED):
    if name == "LR": return LogisticRegression(max_iter=1000, random_state=seed)
    if name == "DT": return DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, class_weight="balanced", random_state=seed)
    if name == "NB": return GaussianNB()
    if name == "RF": return RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_split=20, min_samples_leaf=10,
                                                   class_weight="balanced", random_state=seed, n_jobs=-1)
    if name == "Stacking":
        return StackingClassifier(
            estimators=[("lr", LogisticRegression(C=0.1, max_iter=1000, random_state=seed)),
                        ("dt", DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, class_weight="balanced", random_state=seed)),
                        ("rf", RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=10,
                                                      class_weight="balanced", random_state=seed, n_jobs=-1))],
            final_estimator=LogisticRegression(), n_jobs=-1)
    if name == "SVM": return SVC(kernel="rbf", C=1.0, class_weight="balanced", random_state=seed)
    raise ValueError(name)

def build_pipe(name):
    prep = ColumnTransformer([("te", NestedTargetEncoder(CAT_COLS), CAT_COLS + ["grp"]), ("sc", StandardScaler(), NUM)])
    return Pipeline([("prep", prep), ("model", make_model(name))])

def scores_of(pipe, X):
    m = pipe.named_steps["model"]; Z = pipe.named_steps["prep"].transform(X)
    return m.predict_proba(Z)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(Z)

def metrics(yt, pred, sc=None):
    yt = np.asarray(yt)
    tn, fp, fn, tp = confusion_matrix(yt, pred, labels=[0, 1]).ravel()
    d = dict(n=len(yt), prevalence=float(yt.mean()), acc=(tp + tn) / len(yt),
             phish_rec=tp / max(tp + fn, 1), FPR=fp / max(fp + tn, 1), phish_prec=precision_score(yt, pred, zero_division=0),
             phish_f1=f1_score(yt, pred, zero_division=0), bal_acc=balanced_accuracy_score(yt, pred),
             mcc=matthews_corrcoef(yt, pred), TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn))
    if sc is not None and len(np.unique(yt)) == 2:
        d["roc_auc"] = roc_auc_score(yt, sc); d["pr_auc"] = average_precision_score(yt, sc)
    return d

# ---- training matrix ----
X = df[NUM + CAT_COLS].copy(); X["grp"] = df["dom"]
if TRAIN_FEATURES_EFF == "recomputed":
    t0 = time.time(); X[NUM] = extract_features(df["url"].tolist()).to_numpy()
    X["tld"] = [(url_parts(u)[1] or TLD_MODE) for u in df["url"]]       # same tld definition as the external rows
    print("recomputed train features in", round(time.time() - t0), "s")
tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=SEED).split(X, y, groups=X["grp"]))     # headline split
print("train/test:", len(tr), len(te), "| domain overlap:", len(set(X.iloc[tr].dom) & set(X.iloc[te].dom)))

pipes, internal = {}, []
for m in MODELS:
    t0 = time.time(); p = build_pipe(m).fit(X.iloc[tr], y.iloc[tr]); pipes[m] = p
    pred = p.predict(X.iloc[te]); internal.append({"model": m, **metrics(y.iloc[te], pred, scores_of(p, X.iloc[te]))})
    try: joblib.dump(p, f"{OUT}/models/{m}.joblib")
    except Exception as ex: print("  (could not save", m, ":", ex, ")")
    print(m, "done in", round(time.time() - t0), "s | acc", round(internal[-1]["acc"], 4), "| recall", round(internal[-1]["phish_rec"], 4))
internal = pd.DataFrame(internal); internal.to_csv(OUT + "/internal_test_metrics.csv", index=False)

if TRAIN_FEATURES_EFF == "dataset" and "SVM" in internal.model.values:
    r = internal.set_index("model").loc["SVM"]
    print("\nPaper check, SVM should be TP/TN/FP/FN = 5905/19275/461/339 ->", int(r.TP), int(r.TN), int(r.FP), int(r.FN))
internal.round(4)

train/test: 89251 25980 | domain overlap: 0
SVM done in 85 s | acc 0.9692 | recall 0.9457
LR done in 3 s | acc 0.9435 | recall 0.7921
DT done in 2 s | acc 0.9301 | recall 0.8477
RF done in 6 s | acc 0.9473 | recall 0.8852
Stacking done in 19 s | acc 0.946 | recall 0.8062
NB done in 1 s | acc 0.8514 | recall 0.5195

Paper check, SVM should be TP/TN/FP/FN = 5905/19275/461/339 -> 5905 19275 461 339


,model,n,prevalence,acc,phish_rec,FPR,phish_prec,phish_f1,bal_acc,mcc,TP,TN,FP,FN,roc_auc,pr_auc
0,SVM,25980,0.2403,0.9692,0.9457,0.0234,0.9276,0.9366,0.9612,0.9163,5905,19275,461,339,0.9926,0.9768
1,LR,25980,0.2403,0.9435,0.7921,0.0086,0.9670,0.8709,0.8918,0.8420,4946,19567,169,1298,0.9882,0.9650
2,DT,25980,0.2403,0.9301,0.8477,0.0439,0.8594,0.8535,0.9019,0.8076,5293,18870,866,951,0.9792,0.9329
3,RF,25980,0.2403,0.9473,0.8852,0.0330,0.8946,0.8899,0.9261,0.8553,5527,19085,651,717,0.9875,0.9665
4,Stacking,25980,0.2403,0.9460,0.8062,0.0097,0.9633,0.8778,0.8982,0.8490,5034,19544,192,1210,0.9891,0.9712
5,NB,25980,0.2403,0.8514,0.5195,0.0436,0.7904,0.6270,0.7380,0.5576,3244,18876,860,3000,0.9336,0.7861


## 4. Load the external datasets, evaluate, diagnose


In [6]:
# ---- loading helpers: find files, pick the url / label columns, read only those columns ----
import pyarrow.parquet as pq
URL_CANDS = ["url", "urls", "link", "website"]
LAB_CANDS = ["label", "status", "class", "type", "result"]          # NOT 'target' (PhreshPhish uses it for the brand name)

def _pick(names, cands):
    low = {str(n).lower(): n for n in names}
    for c in cands:
        if c in low: return low[c]
    return None

def list_data_files(d, prefer=None):
    files = sorted({f for e in ("csv", "parquet", "jsonl", "json") for f in glob.glob(f"{d}/**/*.{e}", recursive=True)})
    if prefer:
        pref = [f for f in files if prefer in f.lower()]
        if pref: files = pref
    return files

def read_url_label(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        names = pq.ParquetFile(path).schema.names; u, l = _pick(names, URL_CANDS), _pick(names, LAB_CANDS)
        return None if (u is None or l is None) else pd.read_parquet(path, columns=[u, l]).rename(columns={u: "url", l: "label"})
    if ext in (".jsonl", ".json"):
        d = pd.read_json(path, lines=(ext == ".jsonl")); u, l = _pick(d.columns, URL_CANDS), _pick(d.columns, LAB_CANDS)
        return None if (u is None or l is None) else d[[u, l]].rename(columns={u: "url", l: "label"})
    names = pd.read_csv(path, nrows=0).columns; u, l = _pick(names, URL_CANDS), _pick(names, LAB_CANDS)
    return None if (u is None or l is None) else pd.read_csv(path, usecols=[u, l]).rename(columns={u: "url", l: "label"})

def to_binary(s):
    m = {"1": 1, "true": 1, "phish": 1, "phishing": 1, "malicious": 1, "bad": 1, "unsafe": 1,
         "0": 0, "false": 0, "benign": 0, "legitimate": 0, "legit": 0, "good": 0, "safe": 0}
    out = s.astype(str).str.strip().str.lower().map(m)
    if out.isna().any(): raise ValueError("Unrecognised label values: " + str(s[out.isna()].unique()[:10]) + " -> extend the map in to_binary()")
    return out.astype(int)

def norm_url(u):
    u = str(u).strip().lower().split("://", 1)[-1]
    return (u[4:] if u.startswith("www.") else u).rstrip("/")

def load_external(name, cfg):
    files = list_data_files(cfg["dir"], cfg.get("prefer"))
    print(f"[{name}] files found:", [os.path.relpath(f, cfg["dir"]) for f in files][:12])
    parts = []
    for f in files:
        d = read_url_label(f)
        if d is None: print("   skipped (no url/label columns):", os.path.basename(f)); continue
        print("   using", os.path.basename(f), "| rows", len(d), "| columns -> url, label"); parts.append(d)
    if not parts: raise RuntimeError(f"[{name}] no file with a url column and a label column under {cfg['dir']}")
    raw = pd.concat(parts, ignore_index=True)
    print("   raw label values:", raw["label"].astype(str).value_counts().head(6).to_dict())
    e = pd.DataFrame({"url": raw["url"].astype(str).str.strip(), "label": to_binary(raw["label"])})
    if FLIP.get(name): e["label"] = 1 - e["label"]; print("   !! labels flipped (FLIP)")
    n0 = len(e); e["_n"] = e["url"].map(norm_url); e = e.drop_duplicates("_n")
    train_norm = TRAIN_NORM
    in_train = e["_n"].isin(train_norm); e = e[~in_train].drop(columns="_n").reset_index(drop=True)
    print(f"   {n0} rows -> {len(e)} after dedup and removal of {int(in_train.sum())} URLs that also occur in URL-Phish")
    if len(e) > cfg["max_rows"]: e = e.sample(cfg["max_rows"], random_state=SEED).reset_index(drop=True)
    print("   using", len(e), "rows | phishing share:", round(float(e.label.mean()), 4))
    print("   label=1 examples:", e[e.label == 1].url.head(3).tolist())
    print("   label=0 examples:", e[e.label == 0].url.head(3).tolist())
    return e

TRAIN_NORM = set(df["url"].map(norm_url))
TRAIN_DOMS = set(X.iloc[tr]["dom"])            # domains the models were actually trained on

In [7]:
def evaluate_external(name, cfg):
    od = f"{OUT}/{name}"; os.makedirs(od, exist_ok=True)
    ext = load_external(name, cfg)
    t0 = time.time(); Xe = extract_features(ext["url"].tolist()); parts = ext["url"].map(url_parts)
    Xe["dom"] = [p[0] for p in parts]; Xe["tld"] = [p[1] if p[1] else TLD_MODE for p in parts]; Xe["grp"] = Xe["dom"]
    ye = ext["label"].astype(int); seen = Xe["dom"].isin(TRAIN_DOMS).to_numpy()
    print(f"   features in {round(time.time()-t0)} s | rows whose domain was in the training split: {seen.mean():.4f}")

    rows, rows_unseen, preds, scs = [], [], {}, {}
    for m in MODELS:
        p = pipes[m]; pred = p.predict(Xe); sc = scores_of(p, Xe); preds[m], scs[m] = pred, sc
        rows.append({"model": m, **metrics(ye, pred, sc)})
        if (~seen).sum() > 0 and ye[~seen].nunique() == 2:
            rows_unseen.append({"model": m, **metrics(ye[~seen], pred[~seen], sc[~seen])})
    EXT = pd.DataFrame(rows); EXT.to_csv(f"{od}/external_metrics.csv", index=False)
    EXTU = pd.DataFrame(rows_unseen); EXTU.to_csv(f"{od}/external_metrics_unseen_domains_only.csv", index=False)
    keep = ["phish_rec", "FPR", "phish_prec", "phish_f1", "bal_acc", "roc_auc"]
    cmp = internal.set_index("model")[keep].add_prefix("URLPhish_").join(EXT.set_index("model")[keep].add_prefix("External_"))
    cmp["recall_drop"] = cmp["URLPhish_phish_rec"] - cmp["External_phish_rec"]; cmp["FPR_increase"] = cmp["External_FPR"] - cmp["URLPhish_FPR"]
    cmp.to_csv(f"{od}/comparison_urlphish_vs_external.csv")
    base = []
    for _, r in EXT.iterrows():
        row = {"model": r.model, "recall": r.phish_rec, "FPR": r.FPR}
        for b in BASE_RATES: row[f"precision@{b:.1%}"] = r.phish_rec * b / max(r.phish_rec * b + r.FPR * (1 - b), 1e-12)
        base.append(row)
    BASE = pd.DataFrame(base); BASE.to_csv(f"{od}/base_rate_precision_external.csv", index=False)

    main = "SVM" if "SVM" in MODELS else MODELS[0]
    D = Xe[["is_https", "path_len", "url_len", "entropy"]].copy(); D["y"] = ye.to_numpy(); D["pred"] = preds[main]
    shift = pd.concat({"URLPhish": df.groupby("label")[["is_https", "path_len", "url_len", "entropy"]].mean(),
                       "External": D.groupby("y")[["is_https", "path_len", "url_len", "entropy"]].mean()}, axis=1); shift.to_csv(f"{od}/feature_shift.csv")
    def slice_report(mask_series, nm):
        out = []
        for k, g in D.groupby(mask_series, observed=True):
            out.append({nm: k, "n": len(g), "phish_share": g.y.mean(),
                        "benign_FPR": ((g.pred == 1) & (g.y == 0)).sum() / max((g.y == 0).sum(), 1),
                        "phish_recall": ((g.pred == 1) & (g.y == 1)).sum() / max((g.y == 1).sum(), 1)})
        return pd.DataFrame(out)
    r1 = slice_report(D.is_https.map({0: "http", 1: "https"}), "scheme"); r1.to_csv(f"{od}/diag_by_scheme.csv", index=False)
    r2 = slice_report(pd.cut(D.path_len, [-1, 0, 5, 15, 1e9], labels=["0", "1-5", "6-15", ">15"]), "path_len_bucket"); r2.to_csv(f"{od}/diag_by_pathlen.csv", index=False)
    Xn = Xe.copy(); Xn["is_https"] = float(X.iloc[tr]["is_https"].mean())
    mn = metrics(ye, pipes[main].predict(Xn), scores_of(pipes[main], Xn)); m0 = EXT.set_index("model").loc[main]

    fig, ax = plt.subplots(1, 2, figsize=(11, 4)); x = np.arange(len(MODELS)); w = 0.38
    for a, col, ttl in zip(ax, ["phish_rec", "FPR"], ["Phishing recall", "False-positive rate"]):
        a.bar(x - w/2, internal.set_index("model").loc[MODELS, col], w, label="URL-Phish test"); a.bar(x + w/2, EXT.set_index("model").loc[MODELS, col], w, label=name)
        a.set_xticks(x); a.set_xticklabels(MODELS); a.set_title(ttl); a.legend()
    plt.tight_layout(); plt.savefig(f"{od}/fig_recall_fpr_internal_vs_external.png", dpi=200); plt.close()
    return dict(name=name, ext=ext, EXT=EXT, EXTU=EXTU, cmp=cmp, BASE=BASE, shift=shift, r1=r1, r2=r2, main=main, https_off=(m0, mn), seen=float(seen.mean()))

In [8]:
RESULTS = {}
for nm, cfg in EXT_SETS.items():
    try: RESULTS[nm] = evaluate_external(nm, cfg)
    except Exception as ex: print(f"[{nm}] FAILED:", type(ex).__name__, ex)
print("\ndatasets evaluated:", list(RESULTS))

[Hannousse2020] files found: ['Dataset.csv']
   using Dataset.csv | rows 116600 | columns -> url, label
   raw label values: {'0': 100000, '1': 16600}
   116600 rows -> 0 after dedup and removal of 114283 URLs that also occur in URL-Phish
   using 0 rows | phishing share: nan
   label=1 examples: []
   label=0 examples: []
[Hannousse2020] FAILED: KeyError 'url_len|len'
[PhreshPhish] files found: ['data/training/phreshphish_html_test.parquet']
   using phreshphish_html_test.parquet | rows 168060 | columns -> url, label
   raw label values: {'benign': 91260, 'phish': 76800}
   168060 rows -> 166268 after dedup and removal of 288 URLs that also occur in URL-Phish
   using 30000 rows | phishing share: 0.4566
   label=1 examples: ['http://1win-bf4.xyz/l0/https:%2f%2fs3-kuuu.s3.amazonaws.com%2f3797963.html/1', 'https://tirmgss.lumvia.cfd/helper.php', 'https://oguzhanyasar.av.tr/z0f76a1d14fd21a8fb5fd0d03e0fdc3d3cedae52f?wsidchk=7193543&pdata=https%253a%252f%252foguzhanyasar.av.tr%252fwp-inclu

In [9]:
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50)
summary = []
for nm, R in RESULTS.items():
    main = R["main"]; r = R["cmp"].loc[main]
    print("\n" + "=" * 100); print(f"{nm}  |  rows {len(R['ext'])}  |  phishing share {R['ext'].label.mean():.3f}"); print("=" * 100)
    print("--- URL-Phish test vs external (fixed pipelines) ---"); print(R["cmp"].round(4).to_string())
    print("--- precision implied by external recall/FPR at base rates ---"); print(R["BASE"].round(4).to_string())
    if len(R["EXTU"]): print("--- only rows whose domain was NOT in the training split ---"); print(R["EXTU"][["model", "n", "phish_rec", "FPR", "phish_prec", "phish_f1", "roc_auc"]].round(4).to_string())
    print("--- feature shift (class-wise means) ---"); print(R["shift"].round(3).to_string())
    print(f"--- {main}: by scheme / by path length ---"); print(R["r1"].round(4).to_string()); print(R["r2"].round(4).to_string())
    m0, mn = R["https_off"]; print(f"{main} external recall/FPR original {m0.phish_rec:.4f}/{m0.FPR:.4f}  |  is_https neutralised {mn['phish_rec']:.4f}/{mn['FPR']:.4f}")
    if r.External_roc_auc < 0.5: print(f"\n!!! WARNING: ROC-AUC {r.External_roc_auc:.3f} < 0.5 -> the labels of {nm} are probably INVERTED. Put FLIP = {{'{nm}': True}} in the config cell and rerun the evaluation cells.")
    print("\n=== READ-OUT ===")
    print(f"{main}: recall {r.URLPhish_phish_rec:.3f} -> {r.External_phish_rec:.3f} | FPR {r.URLPhish_FPR:.3f} -> {r.External_FPR:.3f} | ROC-AUC {r.URLPhish_roc_auc:.3f} -> {r.External_roc_auc:.3f}")
    if r.recall_drop < 0.05 and r.FPR_increase < 0.03: print("Small change: the pipeline transfers to this dataset (say 'on this dataset', not 'in deployment').")
    elif r.External_roc_auc > 0.95: print("Ranking still good (high AUC) but the operating point shifted: recall/FPR trade-off changed; threshold needs recalibration.")
    else: print("Large drop: consistent with reliance on URL-Phish source artifacts (HTTPS, short benign paths). Check the scheme/path tables.")
    for m in R["cmp"].index:
        c = R["cmp"].loc[m]; summary.append({"dataset": nm, "model": m, "URLPhish_recall": c.URLPhish_phish_rec, "URLPhish_FPR": c.URLPhish_FPR,
            "External_recall": c.External_phish_rec, "External_FPR": c.External_FPR, "External_precision": c.External_phish_prec,
            "External_F1": c.External_phish_f1, "External_balAcc": c.External_bal_acc, "External_AUC": c.External_roc_auc})
SUMMARY = pd.DataFrame(summary); SUMMARY.to_csv(OUT + "/summary_all_datasets.csv", index=False)
print("\n\n##### SUMMARY (copy this table) #####"); print(SUMMARY.round(4).to_string(index=False))


PhreshPhish  |  rows 30000  |  phishing share 0.457
--- URL-Phish test vs external (fixed pipelines) ---
          URLPhish_phish_rec  URLPhish_FPR  URLPhish_phish_prec  URLPhish_phish_f1  URLPhish_bal_acc  URLPhish_roc_auc  External_phish_rec  External_FPR  External_phish_prec  External_phish_f1  External_bal_acc  External_roc_auc  recall_drop  FPR_increase
model                                                                                                                                                                                                                                                          
SVM                   0.9457        0.0234               0.9276             0.9366            0.9612            0.9926              0.7903        0.4189               0.6132             0.6906            0.6857            0.7616       0.1554        0.3955
LR                    0.7921        0.0086               0.9670             0.8709            0.8918            0.9882        